## Task 3, part 3 - Modelling of the second model using the protein data

This model predicts each perturbation's RNA log2FC fingerprint from that same perturbation's own effect on the 24 measured surface proteins.

In [1]:
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

In [2]:
# load back in the RNA fingerprints and split prepared in Task3_01
DATA_DIR = "/home/ubuntu/data/frangieh"

pert_FC_selected = pd.read_pickle(f"{DATA_DIR}/task3_pert_FC_selected_50.pkl")
train_40 = pd.read_csv(f"{DATA_DIR}/task3_train_40.csv")["perturbation"].tolist()
test_10 = pd.read_csv(f"{DATA_DIR}/task3_test_10.csv")["perturbation"].tolist()

selected_50 = train_40 + test_10
conditions = pert_FC_selected.index.get_level_values("condition").unique().tolist()

## Compute the protein log2FC "fingerprint"

Mirrors exactly what Task3_01 did for RNA: for each condition, compare the mean protein expression of each perturbation's cells to that condition's control cells, on a log2 scale with a pseudocount. 4 of the 24 measured "proteins" are isotype controls (antibody background-binding controls, not real markers) and are dropped, leaving us with 20 real surface markers.

In [19]:
protein = sc.read_h5ad(f"{DATA_DIR}/protein_qc_filtered.h5ad")

# drop isotype controls (here isotype_control in var is nan since they are the control themselfes)
protein = protein[:, protein.var['Isotype_control']!='nan'].copy()

# keep only control cells and cells belonging to one of the 50 selected perturbations
relevant_mask = protein.obs["perturbation"].isin(selected_50) | (protein.obs["perturbation"] == "control")
protein = protein[relevant_mask.values].copy()

# normalize the protein expression data 
sc.pp.normalize_total(protein, target_sum=1e4)
protein_norm = np.asarray(protein.X.todense())

protein.shape

/tmp/ipykernel_56537/1157936093.py:11: UserWarning: Some cells have zero counts
  sc.pp.normalize_total(protein, target_sum=1e4)


(92532, 20)

In [ ]:
# compute the mean protein expression across all control cells under each condition
control_means_protein = {}
for cond in conditions:
    mask = (protein.obs["perturbation_2"] == cond) & (protein.obs["perturbation"] == "control")
    control_means_protein[cond] = protein_norm[mask.values].mean(axis=0)

# log2FC protein vector of 20 elements per perturbation and condition combination, relative to control cells of that condition
protein_FC = {}
# iterate over all conditions and perturbations
for pert in selected_50:
    for cond in conditions:
        # filter for all cells that have that condition and perturbation and average the protein expression across these cells
        mask = (protein.obs["perturbation_2"] == cond) & (protein.obs["perturbation"] == pert)
        pert_mean = protein_norm[mask.values].mean(axis=0)
        # calculate the log2FC with pseudocount for the protein expression in perturbed cells compared to control cells in the same condition
        protein_FC[(pert, cond)] = np.log2((pert_mean + 1) / (control_means_protein[cond] + 1))

# define two level index from the two keys for each FC vector (condition and perturbation)
protein_FC_index = pd.MultiIndex.from_tuples(protein_FC.keys(), names=["perturbation", "condition"])
# convert to dataframe with index = perturbation, condition and columnames = proteins
protein_FC_df = pd.DataFrame(np.vstack(list(protein_FC.values())), index=protein_FC_index, columns=protein.var_names)

protein_FC_df.shape

(150, 20)

## Reduce the target: PCA on the training RNA FC vectors

The RNA fingerprint we want to predict has 2042 values, but we only have 20 protein features and 40 training perturbations per condition. Predicting 2042 outputs from 40 examples would almost certainly overfit, so we compress the target first.

For each condition we fit a PCA on the RNA FC vectors of the 40 training perturbations and keep the first 10 components. Instead of predicting all 2042 genes, the model then only has to predict the 10 PC scores, which are then transformed back into a full FC vector. The PCA is fitted on the training perturbations only, so the 10 held-out perturbations never influence the PCA axis.

In [35]:
n_PCs = 10

# per condition fit PCA on the 40 training genes' RNA FC vectors, and store each training gene's score
target_pca_by_condition = {}
target_scores = {}  # store the scores of the 10 PCs for each training pair of perturbation and condition
for cond in conditions: # iterate over all conditions
    train_fingerprints = np.vstack([pert_FC_selected.loc[(g, cond)].values for g in train_40]) # store RNA FC vectors for each perturbation under the condition in an array (40, 2042)
    pca = PCA(n_components= n_PCs, random_state=42) # create PCA function that keeps 10 components and has a seed for reproducibility
    scores = pca.fit_transform(train_fingerprints) # fit the pca on all training fingerprints and store one row of 10 scores (for all components) for each perturbation in training
    target_pca_by_condition[cond] = pca # store the fitted PCA 
    for gene, score in zip(train_40, scores): # pair each gene name in train 40 and the scores (along the 10 pca components) and iterate over them
        target_scores[(gene, cond)] = score # store the scores in the target_scores dictionary with two keys

# compute how much of the training FC vectors' variance these 10 PCs capture, per condition
{cond: target_pca_by_condition[cond].explained_variance_ratio_.sum() for cond in conditions}

{'Control': np.float32(0.61914194),
 'IFNγ': np.float32(0.6695988),
 'Co-culture': np.float32(0.76921785)}

## Ridge regression prediction

In this step we define a ridge regression to predict the log2FC vector across all 2042 genes for a given perturbation under a condition given a pool of training genes and a regularization strength of alpha (the query is always excluded from the pool so that we can use this function for LOOCV). The input for the regression are the protein FC values across all 20 proteins which are Z-transformed before fitting and the target is defined by the 10 PC scores. After fitting this relationship we can return the PC scores for an unknown perturbation and transform it back into the full FC vector using the PCA for that condition.

In [ ]:
# define a function to predict the gene expression log2FC vector from the protein measurements under a given perturbation in one condition
def ridge_predict(query_gene, condition, alpha, pool_genes): # alpha will need to be selected later, pool genes = training perturbations
    """Predict an RNA fingerprint via Ridge regression from protein log2FC to RNA target-PC scores."""
    # exclude the query gene itself from the pool used to fit the model
    fit_genes = [g for g in pool_genes if g != query_gene] # define all genes used to fit the regression as those which are in the pool but not in the query

    # define input (protein data for a perturbation and condition) and target (scores of the 10 PCs for a perturbation and condition) of the model
    X = np.vstack([protein_FC_df.loc[(g, condition)].values for g in fit_genes]) # store for all perturbation the protein data in an array
    y = np.vstack([target_scores[(g, condition)] for g in fit_genes]) # store for all perturbation the target scores in an array 

    # z-transform the protein features before regularized regression so there is no difference in penalization purely based on the scale of the data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X) # scaler learns and stores the mean and sd of the data used for the transformation so it can be used later to transform the query data and it transforms the X at the same time

    ridge = Ridge(alpha=alpha) # creates a ridge regression model with regularization strenght alpha
    ridge.fit(X_scaled, y) # fit the model on the scaled protein data to model the 10 PC scores of the RNA data 

    query_X = scaler.transform(protein_FC_df.loc[(query_gene, condition)].values.reshape(1, 20)) # grab the protein data (only values) of the query perturbation under the condition and turn it into a 1 x 20 matrix (instead of a normal vector) and scale it according to the pool mean and sd
    pred_scores = ridge.predict(query_X) # return the predicted value of each of teh 10 PCs for the perturbation

    # reconstruct the full RNA fingerprint from the predicted target-PC scores
    pred_fingerprint = target_pca_by_condition[condition].inverse_transform(pred_scores)[0] # transform the 10 PC scores back into the full FC vector by usiing the condition's PCA
    return pred_fingerprint # return the fingerprint(log2FC vector for RNA expression)

## Choosing the Ridge regularization strength (alpha) via leave-one-out cross-validation

Hold out one training gene at a time, predict it from the other 39 (per condition), and compare a few candidate alpha values. This never touches the 10 held-out test genes -- alpha is fixed before we ever look at them.

In [ ]:
candidate_alphas = [0.1, 1, 10, 100, 1000, 10_000, 100_000, 1_000_000]

cv_mse_by_alpha = {}
for alpha in candidate_alphas:
    squared_errors = []
    for cond in conditions:
        for gene in train_40:
            # ridge_predict excludes the query gene itself from the pool, so this is a leave-one-out prediction
            pred = ridge_predict(gene, cond, alpha, train_40)
            true = pert_FC_selected.loc[(gene, cond)].values
            squared_errors.append(np.mean((true - pred) ** 2))
    cv_mse_by_alpha[alpha] = np.mean(squared_errors)

best_alpha = min(cv_mse_by_alpha, key=cv_mse_by_alpha.get)
cv_mse_by_alpha, best_alpha

({0.1: np.float32(0.0070858756),
  1: np.float32(0.005488515),
  10: np.float32(0.003390293),
  100: np.float32(0.0023222358),
  1000: np.float32(0.0021853666),
  10000: np.float32(0.0021873454),
  100000: np.float32(0.0021882053),
  1000000: np.float32(0.0021882995)},
 1000)

## Predict the held-out test genes and evaluate

Use the chosen alpha to predict each of the 10 held-out genes' RNA fingerprint from their own protein log2FC, then evaluate with the same metrics used for the other models so results are directly comparable.

In [8]:
# predict each held-out (gene, condition) pair from the 40 training genes, using the CV-chosen alpha
ridge_predictions = {
    (gene, cond): ridge_predict(gene, cond, best_alpha, train_40)
    for cond in conditions
    for gene in test_10
}


def evaluate_predictions(true_df, predictions_by_row):
    """Compare each true fingerprint against its predicted fingerprint (looked up per row)."""
    records = []
    for (pert, cond), true_fc in true_df.iterrows():
        pred_fc = predictions_by_row[(pert, cond)]
        pearson_r, _ = pearsonr(true_fc, pred_fc)
        spearman_r, _ = spearmanr(true_fc, pred_fc)
        mse = np.mean((true_fc - pred_fc) ** 2)
        records.append({
            "perturbation": pert,
            "condition": cond,
            "pearson_r": pearson_r,
            "spearman_r": spearman_r,
            "mse": mse,
        })
    return pd.DataFrame(records)


ridge_eval = evaluate_predictions(pert_FC_selected.loc[test_10, :], ridge_predictions)
ridge_eval

,perturbation,condition,pearson_r,spearman_r,mse
0,KCNN4,Control,0.828076,0.452951,0.001510
1,KCNN4,IFNγ,0.844792,0.393864,0.001099
2,KCNN4,Co-culture,0.865248,0.340129,0.001129
3,TIMM50,Control,0.740555,0.392573,0.002776
4,TIMM50,IFNγ,0.626393,0.295484,0.003330
5,TIMM50,Co-culture,0.700764,0.198442,0.003850
6,TXNDC17,Control,0.807620,0.494533,0.004335
7,TXNDC17,IFNγ,0.623309,0.439239,0.004303
8,TXNDC17,Co-culture,0.752614,0.365463,0.004496
9,CORO1A,Control,0.804729,0.374306,0.001203


In [9]:
metrics = ["pearson_r", "spearman_r", "mse"]

# per-condition breakdown (n=10 genes each) -- for biological interpretation
per_condition = ridge_eval.groupby("condition")[metrics].agg(["mean", "std"])

# pooled across all held-out (gene, condition) pairs (n=30) -- single headline number, comparable to the other models
overall = ridge_eval[metrics].agg(["mean", "std"])

per_condition

pearson_r           spearman_r                 mse          
                mean       std       mean       std      mean       std
condition                                                              
Co-culture  0.593808  0.498146   0.265838  0.105129  0.003968  0.005932
Control     0.755588  0.135749   0.399165  0.116829  0.002491  0.001575
IFNγ        0.719905  0.251020   0.377741  0.097322  0.002410  0.001890

In [10]:
overall

,pearson_r,spearman_r,mse
mean,0.689767,0.347582,0.002956
std,0.327519,0.118914,0.003651


## Discussion (initial draft -- please rewrite)

**What this notebook does:** Trains a Ridge regression model that predicts a perturbation's RNA fingerprint from that *same* perturbation's own effect on 20 measured surface proteins (log2FC, computed exactly like the RNA fingerprint in Task3_01). This is legitimate because perturbations are only held out from *training*, not from test-time inputs -- so the held-out gene's own protein readout is fair game. As in model 2, the ~2042-dim RNA target is reduced via PCA (fit on the 40 training genes only) to 10 components, and alpha is chosen via leave-one-gene-out cross-validation on the training genes.

**Results:** Unlike model 2, LOOCV finds a genuine interior minimum, at alpha = 1000 (MSE 0.0021854), with clearly worse MSE both below and above it (0.0070859 at alpha=0.1, climbing back up to 0.0021883 by alpha=1,000,000). This is evidence of real, if weak, exploitable signal between a perturbation's protein-level effect and its RNA-level effect -- something model 2's expression statistics never showed any sign of. Final test performance: Pearson r = 0.690, Spearman r = 0.348, MSE = 0.00296 -- numerically almost identical to the baseline and model 2, despite the qualitatively different (and real) signal found during cross-validation. With only 40 training genes, Ridge's regularization has to be strong enough that even genuine signal only pulls predictions slightly away from the training-mean fingerprint.

**A dead end worth documenting:** we also tried training on individual *cells* rather than 40 gene-level averages, motivated by the fact that RNA and protein were measured on the very same physical cells (confirmed by matching cell barcodes across the two datasets), which in principle gives ~36,000 genuinely different (protein log2FC, RNA log2FC) training rows instead of 40. This performed markedly *worse* (Pearson 0.551, Spearman 0.186) rather than better. Our interpretation: pseudobulk averaging isn't just a formality, it's what cancels out heavy single-cell measurement noise on both sides before the regression ever sees the data. Fitting Ridge on individually noisy (X, y) pairs suffers from attenuation bias (noise in the predictor itself biases the fitted relationship toward zero); averaging predictions back together at the end reduces noise in the output, but can't undo bias already baked into the fitted coefficients. We reverted to the gene-level formulation as the reported model 3.